# Sleep Health — KPI Dashboards & Advanced Analytics

Interactive dashboards, statistical hypothesis testing, ML-based feature importance, and executive summary panels.  
**Dataset**: 100,000 individuals × 32 features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/sleep_health_dataset.csv')
print(f'Shape: {df.shape}')
df.head(3)

---
## 1. Executive KPI Dashboard
A high-level view of the most critical sleep health metrics across the entire population.

In [ ]:
def kpi_card(ax, title, value, subtitle, color, icon=''):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    rect = FancyBboxPatch((0.03, 0.05), 0.94, 0.9, boxstyle='round,pad=0.05',
                           facecolor=color, alpha=0.15, edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(0.5, 0.78, title, ha='center', va='center', fontsize=11,
            fontweight='bold', color='#555', transform=ax.transAxes)
    ax.text(0.5, 0.45, f'{icon} {value}', ha='center', va='center', fontsize=26,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.15, subtitle, ha='center', va='center', fontsize=9,
            color='#777', transform=ax.transAxes)

avg_duration = df['sleep_duration_hrs'].mean()
avg_quality = df['sleep_quality_score'].mean()
pct_rested = df['felt_rested'].mean() * 100
avg_stress = df['stress_score'].mean()
severe_risk_pct = (df['sleep_disorder_risk'] == 'Severe').mean() * 100
avg_cognitive = df['cognitive_performance_score'].mean()
avg_latency = df['sleep_latency_mins'].mean()
pct_sleep_aid = df['sleep_aid_used'].mean() * 100

fig, axes = plt.subplots(2, 4, figsize=(20, 7))
fig.suptitle('SLEEP HEALTH KPI DASHBOARD', fontsize=20, fontweight='bold', y=1.02, color='#2c3e50')

kpis = [
    ('Avg Sleep Duration', f'{avg_duration:.1f} hrs', 'Recommended: 7-9 hrs', '#e74c3c', '⏰'),
    ('Avg Sleep Quality',  f'{avg_quality:.1f}/10',   'Population mean',      '#3498db', '⭐'),
    ('Felt Rested',        f'{pct_rested:.1f}%',      'of total population',  '#e67e22', '😴'),
    ('Avg Stress Score',   f'{avg_stress:.1f}/10',    'Lower is better',      '#9b59b6', '🧠'),
    ('Severe Risk',        f'{severe_risk_pct:.1f}%', 'Sleep disorder risk',  '#e74c3c', '⚠️'),
    ('Avg Cognitive Score', f'{avg_cognitive:.0f}',   'Out of 100',           '#2ecc71', '📊'),
    ('Sleep Latency',      f'{avg_latency:.0f} min',  'Time to fall asleep',  '#1abc9c', '⏳'),
    ('Sleep Aid Usage',    f'{pct_sleep_aid:.1f}%',   'Using sleep aids',     '#f39c12', '💊'),
]

for ax, (title, value, sub, color, icon) in zip(axes.flatten(), kpis):
    kpi_card(ax, title, value, sub, color, icon)

plt.tight_layout()
plt.show()

print(f'\nQuick Health Check:')
print(f'  {pct_rested:.0f}% of people feel rested — a potential well-being concern')
print(f'  Average sleep is {avg_duration:.2f} hrs — {7 - avg_duration:.2f} hrs below the 7-hr minimum')
print(f'  {severe_risk_pct:.1f}% face severe sleep disorder risk — needs targeted intervention')
print(f'  Sleep aid adoption at {pct_sleep_aid:.1f}% — indicates widespread sleep difficulties')

---
## 2. Demographic & Occupation Deep-Dive Dashboard
Multi-dimensional segmentation analysis across gender, occupation, and country.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

occ_qual = df.groupby('occupation').agg(
    avg_quality=('sleep_quality_score', 'mean'),
    avg_duration=('sleep_duration_hrs', 'mean'),
    count=('person_id', 'count')
).sort_values('avg_quality', ascending=True)

top_occ = occ_qual.tail(15)
colors_occ = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(top_occ)))
axes[0,0].barh(top_occ.index, top_occ['avg_quality'], color=colors_occ, edgecolor='white')
axes[0,0].set_title('Sleep Quality by Occupation', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Avg Sleep Quality Score')
for i, (val, dur) in enumerate(zip(top_occ['avg_quality'], top_occ['avg_duration'])):
    axes[0,0].text(val + 0.02, i, f'{val:.2f} ({dur:.1f}h)', va='center', fontsize=8)

gender_metrics = df.groupby('gender').agg(
    sleep_quality=('sleep_quality_score', 'mean'),
    sleep_duration=('sleep_duration_hrs', 'mean'),
    stress=('stress_score', 'mean'),
    cognitive=('cognitive_performance_score', 'mean'),
    latency=('sleep_latency_mins', 'mean'),
    rested_pct=('felt_rested', 'mean')
).round(2)

x_labels = ['Sleep Quality', 'Duration (hrs)', 'Stress', 'Cognitive/10', 'Latency (min/5)', 'Rested %x10']
x = np.arange(len(x_labels))
width = 0.25
for i, (gender, row) in enumerate(gender_metrics.iterrows()):
    vals = [row['sleep_quality'], row['sleep_duration'], row['stress'],
            row['cognitive']/10, row['latency']/5, row['rested_pct']*10]
    axes[0,1].bar(x + i*width, vals, width, label=gender, alpha=0.85)
axes[0,1].set_xticks(x + width); axes[0,1].set_xticklabels(x_labels, rotation=15)
axes[0,1].set_title('Key Metrics by Gender', fontsize=14, fontweight='bold')
axes[0,1].legend(); axes[0,1].set_ylabel('Normalized Values')

country_qual = df.groupby('country')['sleep_quality_score'].mean().sort_values()
bottom5 = country_qual.head(5)
top5 = country_qual.tail(5)
combined = pd.concat([bottom5, top5])
colors_country = ['#e74c3c']*5 + ['#2ecc71']*5
axes[1,0].barh(combined.index, combined.values, color=colors_country, edgecolor='white')
axes[1,0].axvline(country_qual.mean(), color='gray', ls='--', lw=1.5, label=f'Global Mean: {country_qual.mean():.2f}')
axes[1,0].set_title('Top & Bottom 5 Countries by Sleep Quality', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Avg Sleep Quality Score'); axes[1,0].legend()

chrono_data = df.groupby('chronotype').agg(
    count=('person_id', 'count'),
    avg_quality=('sleep_quality_score', 'mean'),
    avg_duration=('sleep_duration_hrs', 'mean')
).sort_values('avg_quality', ascending=False)

bars = axes[1,1].bar(chrono_data.index, chrono_data['avg_quality'],
                      color=['#f1c40f', '#e67e22', '#3498db'], edgecolor='white', linewidth=1.5)
axes[1,1].set_title('Sleep Quality by Chronotype', fontsize=14, fontweight='bold')
axes[1,1].set_ylabel('Avg Sleep Quality')
for bar, dur, cnt in zip(bars, chrono_data['avg_duration'], chrono_data['count']):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    f'{bar.get_height():.2f}\n({dur:.1f}h, n={cnt:,})',
                    ha='center', fontsize=9)

plt.suptitle('DEMOGRAPHIC & OCCUPATION DASHBOARD', fontsize=18, fontweight='bold', y=1.01, color='#2c3e50')
plt.tight_layout()
plt.show()

---
## 3. Statistical Hypothesis Testing
Rigorous statistical tests to validate findings — proving patterns aren't just noise.

In [ ]:
from scipy import stats

print('=' * 70)
print('STATISTICAL HYPOTHESIS TESTING (alpha = 0.05)')
print('=' * 70)

exercise_yes = df[df['exercise_day'] == 1]['sleep_quality_score']
exercise_no  = df[df['exercise_day'] == 0]['sleep_quality_score']
t_stat, p_val = stats.ttest_ind(exercise_yes, exercise_no)
cohens_d = (exercise_yes.mean() - exercise_no.mean()) / np.sqrt(
    (exercise_yes.std()**2 + exercise_no.std()**2) / 2)

print(f'\nTEST 1: Exercise vs No-Exercise Sleep Quality')
print(f'   Exercise mean: {exercise_yes.mean():.3f}  |  No-exercise: {exercise_no.mean():.3f}')
print(f'   t-statistic: {t_stat:.4f}  |  p-value: {p_val:.2e}')
print(f"   Cohen's d: {cohens_d:.4f}")
print(f'   Result: {"SIGNIFICANT" if p_val < 0.05 else "Not significant"}')

caffeine_groups = [group['sleep_quality_score'].values
                   for _, group in df.groupby('caffeine_mg_before_bed')]
f_stat, p_anova = stats.f_oneway(*caffeine_groups)
print(f'\nTEST 2: Caffeine Impact (One-Way ANOVA)')
print(f'   F-statistic: {f_stat:.4f}  |  p-value: {p_anova:.2e}')
print(f'   Result: {"SIGNIFICANT" if p_anova < 0.05 else "Not significant"}')

mh_groups = [group['sleep_quality_score'].values
             for _, group in df.groupby('mental_health_condition')]
f_mh, p_mh = stats.f_oneway(*mh_groups)
print(f'\nTEST 3: Mental Health vs Sleep Quality (ANOVA)')
print(f'   F-statistic: {f_mh:.4f}  |  p-value: {p_mh:.2e}')
print(f'   Result: {"SIGNIFICANT" if p_mh < 0.05 else "Not significant"}')

contingency = pd.crosstab(df['shift_work'], df['sleep_disorder_risk'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f'\nTEST 4: Shift Work vs Disorder Risk (Chi-Square)')
print(f'   Chi2: {chi2:.4f}  |  p-value: {p_chi:.2e}  |  DoF: {dof}')
print(f'   Result: {"SIGNIFICANT" if p_chi < 0.05 else "Not significant"}')

r_stress, p_stress = stats.pearsonr(df['stress_score'], df['sleep_quality_score'])
print(f'\nTEST 5: Stress vs Sleep Quality (Pearson)')
print(f'   r = {r_stress:.4f}  |  p-value: {p_stress:.2e}')
print(f'   Result: {"SIGNIFICANT" if p_stress < 0.05 else "Not significant"}')

print('\n' + '=' * 70)

---
## 4. Risk Factor Profiling — Radar Chart & Composite Score
A **Sleep Health Index** (0-100) combining 8 normalized indicators.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

score_cols = {
    'sleep_quality_score': 1,
    'sleep_duration_hrs': 1,
    'deep_sleep_percentage': 1,
    'rem_percentage': 1,
    'sleep_latency_mins': -1,
    'wake_episodes_per_night': -1,
    'stress_score': -1,
    'cognitive_performance_score': 1
}

scaler = MinMaxScaler()
for col, direction in score_cols.items():
    scaled = scaler.fit_transform(df[[col]])
    if direction == -1:
        scaled = 1 - scaled
    df[f'{col}_norm'] = scaled.flatten()

norm_cols = [f'{c}_norm' for c in score_cols.keys()]
df['sleep_health_index'] = (df[norm_cols].mean(axis=1) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

radar_metrics = ['sleep_quality_score', 'sleep_duration_hrs', 'deep_sleep_percentage',
                 'rem_percentage', 'cognitive_performance_score']
radar_labels = ['Sleep Quality', 'Duration', 'Deep Sleep %', 'REM %', 'Cognitive']
risk_groups = ['Healthy', 'Mild', 'Moderate', 'Severe']
risk_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=False).tolist()
angles += angles[:1]

ax_radar = fig.add_subplot(121, polar=True)
for risk, color in zip(risk_groups, risk_colors):
    subset = df[df['sleep_disorder_risk'] == risk]
    values = [subset[f'{m}_norm'].mean() for m in radar_metrics]
    values += values[:1]
    ax_radar.plot(angles, values, 'o-', linewidth=2, label=risk, color=color)
    ax_radar.fill(angles, values, alpha=0.1, color=color)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(radar_labels, fontsize=10)
ax_radar.set_title('Risk Group Profiles (Radar)', fontsize=14, fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
axes[0].axis('off')

for risk, color in zip(risk_groups, risk_colors):
    subset = df[df['sleep_disorder_risk'] == risk]['sleep_health_index']
    axes[1].hist(subset, bins=40, alpha=0.5, label=f'{risk} (mean={subset.mean():.1f})',
                 color=color, edgecolor='white')
axes[1].set_title('Sleep Health Index Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sleep Health Index (0-100)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('RISK FACTOR PROFILING', fontsize=18, fontweight='bold', y=1.02, color='#2c3e50')
plt.tight_layout()
plt.show()

shi_stats = df.groupby('sleep_disorder_risk')['sleep_health_index'].agg(['mean','median','std','min','max'])
shi_stats = shi_stats.reindex(risk_groups).round(1)
print('\nSleep Health Index Summary by Risk Group:')
print(shi_stats.to_string())

---
## 5. Lifestyle Impact — ML Feature Importance
Random Forest model ranking which lifestyle factors matter most for sleep quality.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

feature_cols = ['sleep_duration_hrs', 'rem_percentage', 'deep_sleep_percentage',
                'sleep_latency_mins', 'wake_episodes_per_night', 'caffeine_mg_before_bed',
                'alcohol_units_before_bed', 'screen_time_before_bed_mins', 'exercise_day',
                'steps_that_day', 'nap_duration_mins', 'stress_score', 'work_hours_that_day',
                'heart_rate_resting_bpm', 'bmi', 'room_temperature_celsius', 'shift_work']

X = df[feature_cols].fillna(0)
y = df['sleep_quality_score']

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

colors_imp = plt.cm.viridis(np.linspace(0.2, 0.9, len(importance)))
axes[0].barh(importance.index, importance.values, color=colors_imp, edgecolor='white')
axes[0].set_title('Feature Importance for Sleep Quality\n(Random Forest)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance')
for i, v in enumerate(importance.values):
    axes[0].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=8)

df['quality_tier'] = pd.qcut(df['sleep_quality_score'], q=3, labels=['Low', 'Medium', 'High'])
top6 = importance.tail(6).index.tolist()
tier_comparison = df.groupby('quality_tier')[top6].mean()
tier_comparison_norm = tier_comparison.apply(lambda x: (x - x.min()) / (x.max() - x.min()))

tier_comparison_norm.T.plot(kind='barh', ax=axes[1],
                            color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white')
axes[1].set_title('Top 6 Factors: Low vs Medium vs High Quality Sleepers\n(Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Normalized Mean Value')
axes[1].legend(title='Quality Tier')

plt.suptitle('WHAT IMPACTS SLEEP QUALITY MOST?', fontsize=18, fontweight='bold', y=1.02, color='#2c3e50')
plt.tight_layout()
plt.show()

print(f'\nModel R2 Score: {rf.score(X, y):.4f}')
print(f'\nTop 5 Factors Driving Sleep Quality:')
for i, (feat, imp) in enumerate(importance.tail(5).iloc[::-1].items(), 1):
    print(f'   {i}. {feat.replace("_", " ").title()} — Importance: {imp:.4f}')

---
## 6. Seasonal & Work Pattern Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
season_metrics = df.groupby('season').agg(
    quality=('sleep_quality_score', 'mean'),
    duration=('sleep_duration_hrs', 'mean'),
    rested=('felt_rested', 'mean')
).reindex(season_order)

x = np.arange(len(season_order))
w = 0.25
axes[0,0].bar(x - w, season_metrics['quality'], w, label='Quality', color='#3498db')
axes[0,0].bar(x, season_metrics['duration'], w, label='Duration (hrs)', color='#2ecc71')
axes[0,0].bar(x + w, season_metrics['rested']*10, w, label='Rested %/10', color='#e67e22')
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(season_order)
axes[0,0].set_title('Sleep Metrics by Season', fontsize=14, fontweight='bold')
axes[0,0].legend()

day_metrics = df.groupby('day_type').agg(
    quality=('sleep_quality_score', 'mean'),
    duration=('sleep_duration_hrs', 'mean'),
    stress=('stress_score', 'mean'),
    rested=('felt_rested', 'mean')
)
day_metrics_plot = day_metrics[['quality', 'duration', 'stress']].T
day_metrics_plot.plot(kind='bar', ax=axes[0,1], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[0,1].set_title('Weekday vs Weekend', fontsize=14, fontweight='bold')
axes[0,1].set_ylabel('Mean Value'); axes[0,1].tick_params(axis='x', rotation=0)

df['work_bin'] = pd.cut(df['work_hours_that_day'], bins=[0,4,8,10,12,16],
                         labels=['0-4h','4-8h','8-10h','10-12h','12-16h'])
work_qual = df.groupby('work_bin', observed=True).agg(
    quality=('sleep_quality_score', 'mean'),
    stress=('stress_score', 'mean'))

ax2 = axes[1,0].twinx()
axes[1,0].bar(work_qual.index.astype(str), work_qual['quality'], color='#3498db', alpha=0.7, label='Sleep Quality')
ax2.plot(work_qual.index.astype(str), work_qual['stress'], 'ro-', linewidth=2, markersize=8, label='Stress')
axes[1,0].set_title('Work Hours: Sleep Quality & Stress', fontsize=14, fontweight='bold')
axes[1,0].set_ylabel('Avg Sleep Quality', color='#3498db')
ax2.set_ylabel('Avg Stress Score', color='red')
axes[1,0].legend(loc='upper left'); ax2.legend(loc='upper right')

shift_metrics = df.groupby('shift_work').agg(
    quality=('sleep_quality_score', 'mean'),
    duration=('sleep_duration_hrs', 'mean'),
    disorder_severe=('sleep_disorder_risk', lambda x: (x=='Severe').mean()*100),
    health_index=('sleep_health_index', 'mean')
)
shift_metrics.index = ['No Shift Work', 'Shift Work']
shift_metrics.plot(kind='bar', ax=axes[1,1], color=['#2ecc71','#3498db','#e74c3c','#f39c12'],
                    edgecolor='white', rot=0)
axes[1,1].set_title('Shift Work Impact', fontsize=14, fontweight='bold')
axes[1,1].set_ylabel('Value')
axes[1,1].legend(['Quality', 'Duration', 'Severe Risk %', 'Health Index'], fontsize=9)

plt.suptitle('SEASONAL & WORK PATTERN DASHBOARD', fontsize=18, fontweight='bold', y=1.01, color='#2c3e50')
plt.tight_layout()
plt.show()

---
## 7. Executive Summary Dashboard
Final consolidated view with risk funnel, improvement opportunities, and occupation risk heatmap.

In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[17,25,35,45,55,70],
                         labels=['18-25','26-35','36-45','46-55','56-69'])

fig = plt.figure(figsize=(22, 16))
fig.suptitle('EXECUTIVE SUMMARY DASHBOARD', fontsize=24, fontweight='bold', y=0.98, color='#2c3e50')

ax1 = fig.add_subplot(2, 3, 1)
risk_counts = df['sleep_disorder_risk'].value_counts().reindex(['Healthy','Mild','Moderate','Severe'])
risk_pcts = (risk_counts / len(df) * 100).values
bars = ax1.barh(['Healthy','Mild','Moderate','Severe'], risk_pcts,
                 color=['#2ecc71','#f1c40f','#e67e22','#e74c3c'], edgecolor='white')
for bar, pct, cnt in zip(bars, risk_pcts, risk_counts):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{pct:.1f}% ({cnt:,})', va='center', fontsize=10, fontweight='bold')
ax1.set_title('Population Risk Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('%')

ax2 = fig.add_subplot(2, 3, 2)
age_shi = df.groupby('age_group')['sleep_health_index'].agg(['mean','std'])
ax2.fill_between(range(len(age_shi)), age_shi['mean']-age_shi['std'],
                  age_shi['mean']+age_shi['std'], alpha=0.2, color='#3498db')
ax2.plot(range(len(age_shi)), age_shi['mean'], 'o-', color='#3498db', linewidth=2.5, markersize=10)
ax2.set_xticks(range(len(age_shi))); ax2.set_xticklabels(age_shi.index)
ax2.set_title('Sleep Health Index by Age Group', fontsize=13, fontweight='bold')
ax2.set_ylabel('Health Index (0-100)')

ax3 = fig.add_subplot(2, 3, 3)
risk_factors = {
    'High Caffeine (400mg)': df[df['caffeine_mg_before_bed']==400]['sleep_health_index'].mean(),
    'Shift Work': df[df['shift_work']==1]['sleep_health_index'].mean(),
    'Mental Health: Both': df[df['mental_health_condition']=='Both']['sleep_health_index'].mean(),
    'High Screen Time (2h+)': df[df['screen_time_before_bed_mins']>=120]['sleep_health_index'].mean(),
    'High Stress (8+)': df[df['stress_score']>=8]['sleep_health_index'].mean(),
    'Long Work (12h+)': df[df['work_hours_that_day']>=12]['sleep_health_index'].mean()
}
baseline = df['sleep_health_index'].mean()
rf_sorted = dict(sorted(risk_factors.items(), key=lambda x: x[1]))
colors_rf = ['#e74c3c' if v < baseline else '#2ecc71' for v in rf_sorted.values()]
ax3.barh(list(rf_sorted.keys()), list(rf_sorted.values()), color=colors_rf, edgecolor='white')
ax3.axvline(baseline, color='gray', ls='--', lw=2, label=f'Baseline: {baseline:.1f}')
ax3.set_title('Health Index by Risk Factor', fontsize=13, fontweight='bold')
ax3.set_xlabel('Sleep Health Index'); ax3.legend()

ax4 = fig.add_subplot(2, 3, 4)
improvements = {
    'Exercise Regularly': df[df['exercise_day']==1]['sleep_health_index'].mean() - df[df['exercise_day']==0]['sleep_health_index'].mean(),
    'Reduce Caffeine': df[df['caffeine_mg_before_bed']==0]['sleep_health_index'].mean() - df[df['caffeine_mg_before_bed']==400]['sleep_health_index'].mean(),
    'Reduce Screen Time': df[df['screen_time_before_bed_mins']<=30]['sleep_health_index'].mean() - df[df['screen_time_before_bed_mins']>=120]['sleep_health_index'].mean(),
    'Manage Stress': df[df['stress_score']<=3]['sleep_health_index'].mean() - df[df['stress_score']>=8]['sleep_health_index'].mean(),
    'Treat Mental Health': df[df['mental_health_condition']=='Healthy']['sleep_health_index'].mean() - df[df['mental_health_condition']=='Both']['sleep_health_index'].mean()
}
imp_sorted = dict(sorted(improvements.items(), key=lambda x: x[1]))
bars_imp = ax4.barh(list(imp_sorted.keys()), list(imp_sorted.values()),
                     color=plt.cm.Greens(np.linspace(0.3, 0.9, len(imp_sorted))), edgecolor='white')
for bar in bars_imp:
    ax4.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'+{bar.get_width():.1f} pts', va='center', fontsize=9, fontweight='bold', color='#27ae60')
ax4.set_title('Potential Health Index Improvement', fontsize=13, fontweight='bold')
ax4.set_xlabel('Delta Health Index Points')

ax5 = fig.add_subplot(2, 3, (5, 6))
occ_risk = pd.crosstab(df['occupation'], df['sleep_disorder_risk'], normalize='index') * 100
occ_risk = occ_risk.reindex(columns=['Healthy','Mild','Moderate','Severe'])
occ_risk = occ_risk.sort_values('Severe', ascending=True).tail(15)
sns.heatmap(occ_risk, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax5,
            linewidths=0.5, cbar_kws={'label': '% of Occupation'})
ax5.set_title('Sleep Disorder Risk % by Occupation (Top 15)', fontsize=13, fontweight='bold')
ax5.set_ylabel(''); ax5.set_xlabel('Risk Level')

plt.tight_layout()
plt.show()

---
## Conclusion & Data-Driven Recommendations

### Key Insights Summary

| KPI | Value | Benchmark | Status |
|-----|-------|-----------|--------|
| Avg Sleep Duration | ~6.4 hrs | 7-9 hrs | Below Target |
| Population Felt Rested | ~39% | >70% ideal | Critical |
| Severe Risk Population | ~4% | <1% ideal | Needs Attention |
| Sleep Aid Usage | ~50% | <20% ideal | High Dependency |

### Actionable Recommendations

**1. Workplace Wellness Programs (High Priority)**
- Target occupations with highest severe-risk percentages
- Focus on shift workers (Chi-Square test confirmed association)

**2. Screen Time & Caffeine Policies**
- Progressive quality decline with screen time — recommend screen-free 60min before bed
- 400mg caffeine group scored significantly lower (ANOVA p < 0.05)

**3. Mental Health First**
- 'Both' condition group has the lowest Health Index — largest improvement delta

**4. Exercise Programs**
- Statistically significant sleep quality improvement (t-test confirmed)

**5. Age-Specific Interventions**
- Health Index varies by age group — targeted programs recommended

### Methodology
- **Dataset**: 100,000 individuals x 32 features
- **Statistical Tests**: t-tests, ANOVA, Chi-Square, Pearson (alpha = 0.05)
- **ML Model**: Random Forest for feature importance
- **Composite Metric**: Sleep Health Index (0-100) from 8 normalized indicators

---
*Tools: Python, Pandas, Scikit-learn, Scipy, Matplotlib, Seaborn*